# Data Overview
## OHLCV Price Data Analysis

### Objectives:
- Load and examine ETH/USD OHLCV data
- Understand data structure and types
- Check for missing values and data quality
- Initial statistical summary
- Data range and frequency analysis

### Data Sources:
- CSV file: `../../../data/Eth_OHLCV.csv`
- JSON file: `../../../data/Eth_OHLCV.json`
- SQLite DB: `../../../data/ETH.db`

In [1]:
# ====================================================================
# 📦 IMPORTS AND SETUP
# ====================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# ====================================================================
# 🔧 PATH CONFIGURATION - Find utils folder
# ====================================================================

# Get notebook location
notebook_dir = Path(os.getcwd()).resolve()

def find_ohlcv_root(start_path):
    current = start_path
    for _ in range(5):
        if (current / 'utils').exists() and (current / 'utils' / 'data_loader.py').exists():
            return current
        current = current.parent
    return None

# Find ohlcv root
ohlcv_root = find_ohlcv_root(notebook_dir)

if ohlcv_root:
    utils_path = ohlcv_root / 'utils'
    if str(utils_path) not in sys.path:
        sys.path.insert(0, str(utils_path))
    print(f"✅ OHLCV root: {ohlcv_root}")
    print(f"✅ Utils path: {utils_path}")
else:
    print("⚠️  Could not find ohlcv root. Trying relative path...")
    for rel_path in ['../../utils', '../../../utils', '../../../../utils']:
        test_path = (notebook_dir / rel_path).resolve()
        if test_path.exists() and (test_path / 'data_loader.py').exists():
            if str(test_path) not in sys.path:
                sys.path.insert(0, str(test_path))
            print(f"✅ Found utils at: {test_path}")
            break

# ====================================================================
# 📦 IMPORT UTILITIES
# ====================================================================

try:
    from data_loader import DataLoader
    from visualizations import TradingVisualizer
    from trading_helpers import TradingHelpers
    print("✅ All utilities imported successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("⚠️  Please ensure utils folder exists with required files.")
    raise

print("\n" + "=" * 60)
print("✅ SETUP COMPLETE")
print("=" * 60)

✅ OHLCV root: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv
✅ Utils path: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv\utils
✅ All utilities imported successfully!

✅ SETUP COMPLETE


In [2]:
# Initialize data loader
loader = DataLoader()

# Load data from CSV
df = loader.load_from_csv()

if df.empty:
    print("CSV not found, trying JSON...")
    df = loader.load_from_json()

if df.empty:
    print("JSON not found, trying database...")
    df = loader.load_from_db()

print(f"\n✅ Data loaded successfully!")
print(f"📊 Shape: {df.shape}")
print(f"📅 Date range: {df.index.min()} to {df.index.max()}")
print(f"📈 Total periods: {len(df)}")
df.head(10)

⚠️  CSV file not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\data\Eth_OHLCV.csv
CSV not found, trying JSON...
⚠️  JSON file not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\data\Eth_OHLCV.json
JSON not found, trying database...
⚠️  Database not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\data\ETH.db

✅ Data loaded successfully!
📊 Shape: (0, 0)
📅 Date range: nan to nan
📈 Total periods: 0


""


In [3]:
# Data info
print("=" * 60)
print("📋 DATA INFORMATION")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("📊 DATA TYPES")
print("=" * 60)
print(df.dtypes)

print("\n" + "=" * 60)
print("🔍 NULL VALUES")
print("=" * 60)
print(df.isnull().sum())

📋 DATA INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame

📊 DATA TYPES
Series([], dtype: object)

🔍 NULL VALUES
Series([], dtype: float64)


In [5]:
# Statistical summary
print("=" * 60)
print("📈 STATISTICAL SUMMARY")
print("=" * 60)
df.describe()

📈 STATISTICAL SUMMARY


ValueError: Cannot describe a DataFrame without columns

In [ ]:
# Data quality check
print("=" * 60)
print("🔍 DATA QUALITY CHECK")
print("=" * 60)

# Check for zero or negative values
for col in ['open', 'high', 'low', 'close', 'volume']:
    if col in df.columns:
        invalid = (df[col] <= 0).sum()
        print(f"{col}: {invalid} invalid values (<=0)")

# Check OHLC logic
invalid_ohlc = ((df['high'] < df['low']) | 
               (df['high'] < df['open']) | 
               (df['high'] < df['close']) |
               (df['low'] > df['open']) | 
               (df['low'] > df['close'])).sum()
print(f"\n⚠️ Invalid OHLC relationships: {invalid_ohlc}")

if invalid_ohlc > 0:
    print("\nInvalid rows:")
    invalid_mask = ((df['high'] < df['low']) | 
                   (df['high'] < df['open']) | 
                   (df['high'] < df['close']) |
                   (df['low'] > df['open']) | 
                   (df['low'] > df['close']))
    print(df[invalid_mask].head())